In [1]:
import os
import glob
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

# Filter out specific FutureWarnings
warnings.filterwarnings("ignore", category=FutureWarning, 
                       message="use_inf_as_na option is deprecated")
warnings.filterwarnings("ignore", category=FutureWarning, 
                       message="When grouping with a length-1 list-like")

from causal_evaluation import load_experiment_results, calculate_metrics
from causal_visualization import (
    plot_metrics_comparison,
    plot_error_comparison,
    plot_effect_comparison,
    plot_effect_histogram
)

# Set the main study folder path
main_study_folder = "full_study_results_2025-04-01_15-42-20"

# Create a results folder for the analysis
results_folder = f"analysis_results_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}"
os.makedirs(results_folder, exist_ok=True)

print(f"Loading results from {main_study_folder}...")

# Load parameter studies
print("Loading autocorrelation study...")
auto_results = load_experiment_results(os.path.join(main_study_folder, "auto_study"), "auto")
print("Loading cross-link study...")
cross_results = load_experiment_results(os.path.join(main_study_folder, "cross_study"), "cross")
print("Loading noise study...")
noise_results = load_experiment_results(os.path.join(main_study_folder, "noise_study"), "noise")

# Convert to DataFrames for easier analysis
def results_to_dataframe(results_dict, param_name):
    rows = []
    
    for param_value, effect_dict in results_dict.items():
        for effect_key, effect_data in effect_dict.items():
            # Extract data
            row = {
                'param_name': param_name,
                'param_value': param_value,
                'effect_key': effect_key,
                'true_effect': effect_data.get('true_effect'),
                'pcmci_effect': effect_data.get('pcmci_effect'),
                'bagged_effect': effect_data.get('bagged_effect')
            }
            
            # Add bootstrap statistics if available
            bootstrap_stats = effect_data.get('bootstrap_stats', {})
            if bootstrap_stats:
                row.update({
                    'bootstrap_mean': bootstrap_stats.get('mean'),
                    'bootstrap_std': bootstrap_stats.get('std'),
                    'bootstrap_ci_lower': bootstrap_stats.get('ci_lower'),
                    'bootstrap_ci_upper': bootstrap_stats.get('ci_upper'),
                    'bootstrap_success_rate': bootstrap_stats.get('estimation_success_rate')
                })
                
                # Check if bootstrap_effects is available directly
                if 'bootstrap_effects' in effect_data:
                    row['bootstrap_effects'] = effect_data['bootstrap_effects']
            
            # Add timing information if available
            timings = effect_data.get('timings', {})
            if timings:
                for method, time_val in timings.items():
                    row[f'{method}_time'] = time_val
            
            rows.append(row)
    
    # Create DataFrame
    df = pd.DataFrame(rows)
    
    # Add error columns
    for method in ['pcmci', 'bagged', 'bootstrap']:
        if method == 'bootstrap':
            effect_col = 'bootstrap_mean'
        else:
            effect_col = f'{method}_effect'
        
        # Skip if column doesn't exist
        if effect_col not in df.columns:
            continue
        
        # Calculate errors - handle NaN values safely
        df[f'{method}_error'] = df[effect_col].subtract(df['true_effect'], fill_value=np.nan)
        df[f'{method}_abs_error'] = df[f'{method}_error'].abs()
        df[f'{method}_squared_error'] = df[f'{method}_error'].pow(2)
        
        # Flag if true effect is within CI (for bootstrap)
        if method == 'bootstrap' and 'bootstrap_ci_lower' in df.columns and 'bootstrap_ci_upper' in df.columns:
            df['ci_covers_true'] = (df['true_effect'] >= df['bootstrap_ci_lower']) & \
                                  (df['true_effect'] <= df['bootstrap_ci_upper'])
    
    return df

# Convert to DataFrames
auto_df = results_to_dataframe(auto_results, "auto")
cross_df = results_to_dataframe(cross_results, "cross")
noise_df = results_to_dataframe(noise_results, "noise")

# Combine all results
all_results = pd.concat([auto_df, cross_df, noise_df], ignore_index=True)

# Save the processed data
all_results.to_csv(os.path.join(results_folder, "all_results.csv"), index=False)

# Print basic information
print(f"Loaded {len(all_results)} effect estimations")
print(f"Parameters: {all_results['param_name'].unique()}")
print(f"Parameter values: {sorted(all_results['param_value'].unique())}")
print(f"Effect pairs: {all_results['effect_key'].unique()}")

# Calculate metrics for each parameter study
auto_metrics = calculate_metrics(auto_df, group_by='param_value')
cross_metrics = calculate_metrics(cross_df, group_by='param_value')
noise_metrics = calculate_metrics(noise_df, group_by='param_value')

# Save metrics to CSV
auto_metrics.to_csv(os.path.join(results_folder, "auto_metrics.csv"), index=False)
cross_metrics.to_csv(os.path.join(results_folder, "cross_metrics.csv"), index=False)
noise_metrics.to_csv(os.path.join(results_folder, "noise_metrics.csv"), index=False)

# ------------------------------------------------------------------
# Generate plots
# ------------------------------------------------------------------
print("Generating MAE comparison plots...")

# Autocorrelation
fig = plot_metrics_comparison(
    auto_metrics, 
    param_name='param_value',
    metric='mae', 
    methods=['pcmci', 'bagged', 'bootstrap'],
    save_path=os.path.join(results_folder, "auto_mae_comparison.png")
)
plt.close(fig)

# Cross-link strength
fig = plot_metrics_comparison(
    cross_metrics, 
    param_name='param_value',
    metric='mae', 
    methods=['pcmci', 'bagged', 'bootstrap'],
    save_path=os.path.join(results_folder, "cross_mae_comparison.png")
)
plt.close(fig)

# Noise level
fig = plot_metrics_comparison(
    noise_metrics, 
    param_name='param_value',
    metric='mae', 
    methods=['pcmci', 'bagged', 'bootstrap'],
    save_path=os.path.join(results_folder, "noise_mae_comparison.png")
)
plt.close(fig)

print("Generating RMSE comparison plots...")

# Autocorrelation
fig = plot_metrics_comparison(
    auto_metrics, 
    param_name='param_value',
    metric='rmse', 
    methods=['pcmci', 'bagged', 'bootstrap'],
    save_path=os.path.join(results_folder, "auto_rmse_comparison.png")
)
plt.close(fig)

# Cross-link strength
fig = plot_metrics_comparison(
    cross_metrics, 
    param_name='param_value',
    metric='rmse', 
    methods=['pcmci', 'bagged', 'bootstrap'],
    save_path=os.path.join(results_folder, "cross_rmse_comparison.png")
)
plt.close(fig)

# Noise level
fig = plot_metrics_comparison(
    noise_metrics, 
    param_name='param_value',
    metric='rmse', 
    methods=['pcmci', 'bagged', 'bootstrap'],
    save_path=os.path.join(results_folder, "noise_rmse_comparison.png")
)
plt.close(fig)

print("Generating effect comparison plots...")

# Autocorrelation
fig = plot_effect_comparison(
    auto_df, 
    param_name='auto',
    save_path=os.path.join(results_folder, "auto_effect_comparison.png")
)
plt.close(fig)

# Cross-link strength
fig = plot_effect_comparison(
    cross_df, 
    param_name='cross',
    save_path=os.path.join(results_folder, "cross_effect_comparison.png")
)
plt.close(fig)

# Noise level
fig = plot_effect_comparison(
    noise_df, 
    param_name='noise',
    save_path=os.path.join(results_folder, "noise_effect_comparison.png")
)
plt.close(fig)

print("Generating error comparison plots...")

# Autocorrelation
fig = plot_error_comparison(
    auto_df, 
    param_name='auto',
    error_type='abs',
    save_path=os.path.join(results_folder, "auto_error_comparison.png")
)
plt.close(fig)

# Cross-link strength
fig = plot_error_comparison(
    cross_df, 
    param_name='cross',
    error_type='abs',
    save_path=os.path.join(results_folder, "cross_error_comparison.png")
)
plt.close(fig)

# Noise level
fig = plot_error_comparison(
    noise_df, 
    param_name='noise',
    error_type='abs',
    save_path=os.path.join(results_folder, "noise_error_comparison.png")
)
plt.close(fig)

print("Generating bias comparison plots...")

# Autocorrelation
fig = plot_error_comparison(
    auto_df, 
    param_name='auto',
    error_type='raw',
    save_path=os.path.join(results_folder, "auto_bias_comparison.png")
)
plt.close(fig)

# Cross-link strength
fig = plot_error_comparison(
    cross_df, 
    param_name='cross',
    error_type='raw',
    save_path=os.path.join(results_folder, "cross_bias_comparison.png")
)
plt.close(fig)

# Noise level
fig = plot_error_comparison(
    noise_df, 
    param_name='noise',
    error_type='raw',
    save_path=os.path.join(results_folder, "noise_bias_comparison.png")
)
plt.close(fig)

# ------------------------------------------------------------------
# Plot CI coverage
# ------------------------------------------------------------------
print("Analyzing bootstrap confidence interval coverage...")

# Calculate CI coverage rates by parameter value
if 'ci_covers_true' in auto_df.columns:
    auto_ci_coverage = auto_df.groupby('param_value')['ci_covers_true'].mean()
    cross_ci_coverage = cross_df.groupby('param_value')['ci_covers_true'].mean()
    noise_ci_coverage = noise_df.groupby('param_value')['ci_covers_true'].mean()

    # Plot CI coverage
    plt.figure(figsize=(12, 8))
    plt.plot(auto_ci_coverage.index, auto_ci_coverage.values, marker='o', label='Autocorrelation')
    plt.plot(cross_ci_coverage.index, cross_ci_coverage.values, marker='s', label='Cross-link Strength')
    plt.plot(noise_ci_coverage.index, noise_ci_coverage.values, marker='^', label='Noise Level')
    plt.axhline(y=0.95, color='r', linestyle='--', label='Expected 95% coverage')
    plt.xlabel('Parameter Value')
    plt.ylabel('CI Coverage Rate')
    plt.title('Bootstrap 95% CI Coverage Rate')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(results_folder, "ci_coverage_analysis.png"), dpi=300, bbox_inches='tight')
    plt.close()

    # Save CI coverage data
    ci_coverage_df = pd.DataFrame({
        'param_value': auto_ci_coverage.index,
        'auto_coverage': auto_ci_coverage.values,
        'cross_coverage': cross_ci_coverage.values,
        'noise_coverage': noise_ci_coverage.values
    })
    ci_coverage_df.to_csv(os.path.join(results_folder, "ci_coverage.csv"), index=False)

# ------------------------------------------------------------------
# Plot bootstrap distributions for interesting cases
# ------------------------------------------------------------------
print("Generating bootstrap distribution examples...")

# Select interesting parameter values for each study
interesting_cases = [
    (auto_df, 'auto', auto_df['param_value'].min()),  # Low autocorrelation
    (auto_df, 'auto', auto_df['param_value'].max()),  # High autocorrelation
    (noise_df, 'noise', noise_df['param_value'].min()),  # Low noise
    (noise_df, 'noise', noise_df['param_value'].max())   # High noise
]

for df, param_name, param_value in interesting_cases:
    # Find the closest value in the dataframe
    closest_idx = (df['param_value'] - param_value).abs().idxmin()
    closest_row = df.loc[closest_idx]
    
    if 'bootstrap_effects' in closest_row and isinstance(closest_row['bootstrap_effects'], list):
        # Plot bootstrap distribution
        title = f"{param_name.capitalize()} = {closest_row['param_value']:.2f}"
        fig = plot_effect_histogram(
            closest_row['bootstrap_effects'],
            true_effect=closest_row['true_effect'],
            save_path=os.path.join(results_folder, f"bootstrap_dist_{param_name}_{closest_row['param_value']:.2f}.png")
        )
        plt.close(fig)

# ------------------------------------------------------------------
# Generate LaTeX tables
# ------------------------------------------------------------------
print("Generating LaTeX tables...")

# Function to create LaTeX table from metrics DataFrame
def create_latex_table(metrics_df, param_name):
    # Parameter labels
    param_labels = {
        'auto': 'Autocorrelation',
        'cross': 'Cross-link Strength',
        'noise': 'Noise Level'
    }
    
    # Select representative rows (min, median, max)
    values = sorted(metrics_df['param_value'].unique())
    selected_values = [values[0], values[len(values)//2], values[-1]]
    selected_metrics = metrics_df[metrics_df['param_value'].isin(selected_values)]
    
    # Create LaTeX table
    latex_table = f"\\begin{{table}}[htbp]\n"
    latex_table += f"\\centering\n"
    latex_table += f"\\caption{{Effect Estimation Performance for Varying {param_labels.get(param_name, param_name)}}}\n"
    latex_table += f"\\label{{tab:{param_name}_performance}}\n"
    latex_table += f"\\begin{{tabular}}{{l|ccc|ccc|c}}\n"
    latex_table += f"\\hline\n"
    latex_table += f"{param_labels.get(param_name, param_name)} & \\multicolumn{{3}}{{c|}}{{MAE}} & \\multicolumn{{3}}{{c|}}{{RMSE}} & CI Coverage \\\\\n"
    latex_table += f" Value & PCMCI & Bagged & Bootstrap & PCMCI & Bagged & Bootstrap & Bootstrap \\\\\n"
    latex_table += f"\\hline\n"
    
    for _, row in selected_metrics.iterrows():
        latex_table += f"{row['param_value']:.2f} & "
        latex_table += f"{row.get('pcmci_mae', float('nan')):.4f} & {row.get('bagged_mae', float('nan')):.4f} & {row.get('bootstrap_mae', float('nan')):.4f} & "
        latex_table += f"{row.get('pcmci_rmse', float('nan')):.4f} & {row.get('bagged_rmse', float('nan')):.4f} & {row.get('bootstrap_rmse', float('nan')):.4f} & "
        
        if 'bootstrap_ci_coverage' in row:
            latex_table += f"{row['bootstrap_ci_coverage']*100:.1f}\\% \\\\\n"
        else:
            latex_table += f"N/A \\\\\n"
    
    latex_table += f"\\hline\n"
    latex_table += f"\\end{{tabular}}\n"
    latex_table += f"\\end{{table}}\n"
    
    return latex_table

# Generate and save tables
auto_table = create_latex_table(auto_metrics, 'auto')
cross_table = create_latex_table(cross_metrics, 'cross')
noise_table = create_latex_table(noise_metrics, 'noise')

with open(os.path.join(results_folder, "auto_table.tex"), 'w') as f:
    f.write(auto_table)
    
with open(os.path.join(results_folder, "cross_table.tex"), 'w') as f:
    f.write(cross_table)
    
with open(os.path.join(results_folder, "noise_table.tex"), 'w') as f:
    f.write(noise_table)

# Create consolidated table with overall results
def create_consolidated_table():
    # Calculate overall metrics for each parameter study
    overall_metrics = pd.DataFrame([
        {'Parameter': 'Autocorrelation', **calculate_metrics(auto_df).iloc[0]},
        {'Parameter': 'Cross-link Strength', **calculate_metrics(cross_df).iloc[0]},
        {'Parameter': 'Noise Level', **calculate_metrics(noise_df).iloc[0]}
    ])
    
    # Calculate improvement percentages
    for i, row in overall_metrics.iterrows():
        pcmci_mae = row.get('pcmci_mae', 0)
        bagged_mae = row.get('bagged_mae', 0)
        bootstrap_mae = row.get('bootstrap_mae', 0)
        
        if pcmci_mae > 0:
            overall_metrics.loc[i, 'bagged_mae_imp'] = 100 * (pcmci_mae - bagged_mae) / pcmci_mae
            overall_metrics.loc[i, 'bootstrap_mae_imp'] = 100 * (pcmci_mae - bootstrap_mae) / pcmci_mae
    
    # Create LaTeX table
    latex_table = f"\\begin{{table}}[htbp]\n"
    latex_table += f"\\centering\n"
    latex_table += f"\\caption{{Summary of Effect Estimation Performance Across Parameter Studies}}\n"
    latex_table += f"\\label{{tab:overall_performance}}\n"
    latex_table += f"\\begin{{tabular}}{{l|ccc|cc|c}}\n"
    latex_table += f"\\hline\n"
    latex_table += f"Parameter & \\multicolumn{{3}}{{c|}}{{MAE}} & \\multicolumn{{2}}{{c|}}{{Improvement (\\%)}} & CI Coverage \\\\\n"
    latex_table += f"Study & PCMCI & Bagged & Bootstrap & Bagged & Bootstrap & Bootstrap \\\\\n"
    latex_table += f"\\hline\n"
    
    for _, row in overall_metrics.iterrows():
        latex_table += f"{row['Parameter']} & "
        latex_table += f"{row.get('pcmci_mae', float('nan')):.4f} & {row.get('bagged_mae', float('nan')):.4f} & {row.get('bootstrap_mae', float('nan')):.4f} & "
        latex_table += f"{row.get('bagged_mae_imp', float('nan')):.1f}\\% & {row.get('bootstrap_mae_imp', float('nan')):.1f}\\% & "
        
        if 'bootstrap_ci_coverage' in row:
            latex_table += f"{row['bootstrap_ci_coverage']*100:.1f}\\% \\\\\n"
        else:
            latex_table += f"N/A \\\\\n"
    
    latex_table += f"\\hline\n"
    latex_table += f"\\end{{tabular}}\n"
    latex_table += f"\\end{{table}}\n"
    
    return latex_table, overall_metrics

# Generate and save consolidated table
consolidated_table, overall_metrics = create_consolidated_table()
with open(os.path.join(results_folder, "consolidated_table.tex"), 'w') as f:
    f.write(consolidated_table)

# Save overall metrics
overall_metrics.to_csv(os.path.join(results_folder, "overall_metrics.csv"), index=False)

print(f"\nAnalysis complete! Results saved to {results_folder}/")

Loading results from full_study_results_2025-04-01_15-42-20...
Loading autocorrelation study...
Loading cross-link study...
Loading noise study...
Loaded 27 effect estimations
Parameters: ['auto' 'cross' 'noise']
Parameter values: [np.float64(0.1), np.float64(0.2), np.float64(0.30000000000000004), np.float64(0.3375), np.float64(0.4), np.float64(0.5), np.float64(0.575), np.float64(0.6), np.float64(0.7000000000000001), np.float64(0.8), np.float64(0.8124999999999999), np.float64(0.9), np.float64(1.05), np.float64(1.2875), np.float64(1.525), np.float64(1.7625), np.float64(2.0)]
Effect pairs: ['0_-2_to_3_0']
Generating MAE comparison plots...
Generating RMSE comparison plots...
Generating effect comparison plots...
Generating error comparison plots...
Generating bias comparison plots...
Analyzing bootstrap confidence interval coverage...
Generating bootstrap distribution examples...
Generating LaTeX tables...


KeyError: "Column(s) ['bagged_bias', 'bagged_mae', 'bagged_rmse', 'bootstrap_bias', 'bootstrap_ci_coverage', 'bootstrap_mae', 'bootstrap_rmse', 'pcmci_bias', 'pcmci_mae', 'pcmci_rmse'] do not exist"